# Stag CM Inference

This notebook demonstrates inference with StagCM, a state-gated recurrent model trained on SONAR sentence embeddings from Wikipedia. 

We'll load the trained model, convert input text to 1024-dimensional embeddings using SONAR, generate continuations autoregressively, and convert the predicted embeddings back to text. 

In [11]:
import sys
import os

root_dir = "/home/hukami/Workspace/Gen-AI-Lab"
sys.path.append(root_dir)

import torch
import torch.nn.functional as F
from sonar.inference_pipelines.text import (
    TextToEmbeddingModelPipeline,
    EmbeddingToTextModelPipeline
)
from lit.models.stag_cm import StagCM
import numpy as np

# Check device availability and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Initialize SONAR Models

In [12]:
# Text to embeddings (1024-dim vectors)
text2vec = TextToEmbeddingModelPipeline(
    encoder="text_sonar_basic_encoder", 
    tokenizer="text_sonar_basic_encoder"
)

# Embeddings back to text
vec2text = EmbeddingToTextModelPipeline(
    decoder="text_sonar_basic_decoder", 
    tokenizer="text_sonar_basic_encoder"
)

#### Initialize StagCM Model

In [34]:
checkpoint_dir = "lightning_logs/stag_cm_sonar_experiment/version_7/checkpoints"
checkpoint_file = "epoch=0-step=5000.ckpt"

# Model config (from training)
model_config = {
    "model_dim": 1024,
    "state_dim": 2048,
    "num_heads": 32,
    "num_layers": 1,
}

model = StagCM.load_from_checkpoint(
    os.path.join(
        root_dir,
        checkpoint_dir,
        checkpoint_file,
    )
)
model = model.to(device)


In [19]:
def generate_continuation(model, input_embeddings, max_new_tokens=5):
    """
    Generate sequence continuation autoregressively
    
    Args:
        model: Trained StagCM model
        input_embeddings: (1, seq_len, 1024) tensor
        max_new_tokens: Number of tokens to generate
    """
    device = next(model.parameters()).device
    current_sequence = input_embeddings.to(device)
    
    model.eval()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Get prediction for next token
            output = model(current_sequence)
            next_token = output[:, -1:, :]  # Last predicted token
            
            # Append to sequence
            current_sequence = torch.cat([current_sequence, next_token], dim=1)
    
    return current_sequence[:, input_embeddings.shape[1]:, :]  # Return only new tokens

In [23]:
def stagcm_inference(input_text, model, max_new_tokens=3):
    """Complete pipeline: text → embeddings → StagCM → text"""

    # Step 1: Text to embeddings
    print(f"Input: {input_text}")
    input_embeddings = text2vec.predict([input_text], source_lang="eng_Latn")
    input_tensor = torch.tensor(input_embeddings).unsqueeze(0)  # (1, 1, 1024)
    
    # Step 2: Generate continuation
    print("Generating continuation...")
    generated_embeddings = generate_continuation(model, input_tensor, max_new_tokens)
    
    generated_embeddings = generated_embeddings.cpu()

    # Step 3: Convert back to text
    generated_text = []
    for i in range(generated_embeddings.shape[1]):
        emb = generated_embeddings[0, i, :]
        text = vec2text.predict([emb], target_lang="eng_Latn")[0]
        generated_text.append(text)
    
    print(f"Generated: {' '.join(generated_text)}")
    return generated_text


In [28]:
wikipedia_style_inputs = {
    "Science & Technology":
    [
        "Artificial intelligence refers to the simulation of human intelligence in machines.",
        "Machine learning algorithms enable computers to learn from data without explicit programming.",
        "Deep learning networks consist of multiple layers of interconnected nodes.",
        "Neural networks are inspired by the structure and function of biological neurons.",
        "Natural language processing allows computers to understand and generate human language."
    ],
    
    "History":
    [
        "The Renaissance was a cultural movement that began in Italy during the 14th century.",
        "Leonardo da Vinci was a prominent figure known for his contributions to art and science.",
        "The printing press invention by Johannes Gutenberg revolutionized information dissemination.",
        "Renaissance art emphasized realism, perspective, and human anatomy studies.",
        "This period marked the transition from medieval to early modern Europe."
    ],
    
    "Geography & Nature":
    [
        "The Amazon rainforest spans across nine countries in South America.",
        "It contains approximately 10% of all known species on Earth.",
        "The Amazon River is the second longest river in the world.",
        "Deforestation threatens the biodiversity of this crucial ecosystem.",
        "Indigenous communities have lived in the Amazon for thousands of years."
    ],
    
    "Physics & Astronomy":
    [
        "Black holes are regions of spacetime where gravity is extremely strong.",
        "Nothing, not even light, can escape from within a black hole's event horizon.",
        "Stephen Hawking proposed that black holes emit radiation due to quantum effects.",
        "Supermassive black holes exist at the centers of most galaxies.",
        "The first direct image of a black hole was captured in 2019."
    ],
    
    "Biology & Medicine":
    [
        "DNA contains the genetic instructions for the development of living organisms.",
        "The double helix structure was discovered by Watson, Crick, Franklin, and Wilkins.",
        "Genes are specific sequences of DNA that encode for particular proteins.",
        "Mutations in DNA can lead to genetic disorders or evolutionary adaptations.",
        "CRISPR technology allows scientists to edit genes with unprecedented precision."
    ],
    
    "Ancient Civilizations":
    [
        "Ancient Egypt was one of the world's earliest and longest-lasting civilizations.",
        "The pyramids of Giza were built during the Fourth Dynasty of the Old Kingdom.",
        "Hieroglyphics served as the formal writing system of ancient Egypt.",
        "The Nile River was crucial for agriculture, transportation, and trade.",
        "Pharaohs were considered divine rulers with absolute power over the kingdom."
    ],
    
    "Climate & Environment":
    [
        "Climate change refers to long-term shifts in global temperatures and weather patterns.",
        "Greenhouse gases trap heat in Earth's atmosphere, causing global warming.",
        "Carbon dioxide levels have increased significantly since the Industrial Revolution.",
        "Rising sea levels threaten coastal communities and island nations worldwide.",
        "Renewable energy sources offer sustainable alternatives to fossil fuels."
    ],
    
    "Computer Science":
    [
        "The Internet is a global network of interconnected computer systems.",
        "TCP/IP protocols enable reliable communication between different networks.",
        "Tim Berners-Lee invented the World Wide Web in 1989.",
        "Search engines use complex algorithms to index and rank web pages.",
        "Social media platforms have transformed how people communicate and share information."
    ]
}


In [36]:
def stagcm_sequence_inference(sentences, model, max_new_tokens=5):
    """
    Process entire sentence sequence, then generate continuation
    
    Args:
        sentences: List of sentences (e.g., 5 sentences from Wikipedia)
        model: Trained StagCM model
        max_new_tokens: Number of new sentences to generate
    """
    
    # Get embeddings for all sentences
    input_embeddings = text2vec.predict(sentences, source_lang="eng_Latn")
    # Shape: (num_sentences, 1024) -> (1, num_sentences, 1024)
    input_tensor = torch.tensor(input_embeddings, dtype=torch.float32).unsqueeze(0)
    
    # Step 2: Generate continuation from the full sequence
    print(f"\nGenerating {max_new_tokens} continuation sentences...")
    generated_embeddings = generate_continuation(model, input_tensor, max_new_tokens)
    
    # Step 3: Convert generated embeddings back to text
    generated_embeddings_cpu = generated_embeddings.cpu()
    generated_sentences = []
    
    for i in range(generated_embeddings_cpu.shape[1]):
        emb = generated_embeddings_cpu[0, i, :]
        sentence = vec2text.predict([emb], target_lang="eng_Latn")[0]
        generated_sentences.append(sentence)
    

    return generated_sentences

In [ ]:
# sentences = wikipedia_style_inputs["Science & Technology"]
sentences = [
    "Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making.",
    "It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.",
    "Various subfields of AI research are centered around particular goals and the use of particular tools.",
    "The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, perception, and support for robotics.",
    "To reach these goals, AI researchers have adapted and integrated a wide range of techniques, including search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.",
    "Some companies aim to create artificial general intelligence (AGI)—AI that can complete virtually any cognitive task at least as well as a human.",
    ]

print("Input sequence:")
for i, sentence in enumerate(sentences, 1):
    print(f"  {i}. {sentence}")

generated_sentences = stagcm_sequence_inference(sentences, model, max_new_tokens=5)

print("\nGenerated continuation:")
for i, sentence in enumerate(generated_sentences, len(sentences) + 1):
    print(f"  {i}. {sentence}")
    

Input sequence:
  1. Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making.
  2. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
  3. Various subfields of AI research are centered around particular goals and the use of particular tools.
  4. The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, perception, and support for robotics.
  5. To reach these goals, AI researchers have adapted and integrated a wide range of techniques, including search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operati

/tmp/ipykernel_311350/2581886425.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_tensor = torch.tensor(input_embeddings, dtype=torch.float32).unsqueeze(0)



Generating 5 continuation sentences...

Generated continuation:
  7. Information and communication technology (ICT) is integrated into information and communication technology (ICT).
  8. The data analysis and analysis methodology is based on the use of data analysis and analysis methods.
  9. The data analysis methodology is based on the use of data analytics.
  10. Information and communication technologies and information and communication technologies
  11. Accounting standards and standards for data collection


: 